In [1]:
import contextlib
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
import wandb
from tqdm.autonotebook import tqdm

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.experiments.multimodal_gaussian.canonical import (
    get_canonical_chart_transport_configs,
    get_canonical_chart_transport_monitor_configs,
)
from src.experiments.multimodal_gaussian.config import MultimodalGaussianTrainingConfig
from src.experiments.multimodal_gaussian.state import MultimodalTrainingRuntime

/tmp/ipykernel_3566853/587479404.py:11: TqdmExperimentalWarning:

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)



In [2]:
num_modes = 8
latent_dimension = 8
ambient_dimension = 8
experiment_name = f"{datetime.now():%m%d%H%M%S}-{uuid4().hex[:6]}"
wandb_project = "diffusive-latent-generation"

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=0.1,
    offset=0.0,
    ambient_dimension=ambient_dimension,
)

training_config = MultimodalGaussianTrainingConfig.initialize(
    seed=0,
    train_batch_size=4096,
    eval_batch_size=4096,
    integrated_n_steps=1_000_000,
    chart_transport_config=get_canonical_chart_transport_configs(
        data_config=data_config,
        latent_dimension=latent_dimension,
    ),
    monitor_config=get_canonical_chart_transport_monitor_configs(),
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian")
        / experiment_name
    ),
)

training_config.visualize()

In [3]:
cuda_device = 1
runtime = MultimodalTrainingRuntime.initialize(
    tc=training_config,
    cuda_device=cuda_device,
    wandb_project=wandb_project,
    run_name=experiment_name,
)

Using bfloat16 Automatic Mixed Precision (AMP)
Seed set to 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nlyu/.netrc.
wandb: Currently logged in as: lyuxingjian (lyuxingjian-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Pretrain

Chart pretrain is now delegated to the runtime. It logs constraint and conditioning monitors on the configured schedule.


In [4]:
latest_pretrain_metrics = runtime.chart_pretrain()
latest_pretrain_metrics

pretrain:   0%|          | 0/2000 [00:00<?, ?it/s]

[pretrain] step 1/2000: train: chart_loss=0.7003, data_cycle_loss=0.1558, prior_cycle_loss=0.5444, zero_mean_loss=0.0099, latent_norm_loss=0.5247; monitor: constraint_reconstruction_mean=3.6136, constraint_reconstruction_max=5.4479, constraint_latent_norm_mean=7.9615, encoder_conditioning_mean=6.8700, encoder_conditioning_max=11.3863, decoder_conditioning_mean=1.0103, decoder_conditioning_max=1.0913
[pretrain] step 2000/2000: train: chart_loss=0.0004, data_cycle_loss=0.0001, prior_cycle_loss=0.0002, zero_mean_loss=0.0007, latent_norm_loss=0.7217; monitor: constraint_reconstruction_mean=0.0413, constraint_reconstruction_max=0.1226, constraint_latent_norm_mean=1.2160, encoder_conditioning_mean=1.6686, encoder_conditioning_max=2.1071, decoder_conditioning_mean=1.0358, decoder_conditioning_max=1.2893


{'train_chart_loss': 0.000390430330298841,
 'train_data_cycle_loss': 0.00013810316158924252,
 'train_prior_cycle_loss': 0.00018015304522123188,
 'train_zero_mean_loss': 0.0006885528564453125,
 'train_latent_norm_loss': 0.7217411398887634,
 'monitor_constraint_reconstruction_mean': 0.04133222997188568,
 'monitor_constraint_reconstruction_max': 0.12260916084051132,
 'monitor_constraint_latent_norm_mean': 1.2160381078720093,
 'monitor_encoder_conditioning_mean': 1.6685655117034912,
 'monitor_encoder_conditioning_max': 2.107055187225342,
 'monitor_decoder_conditioning_mean': 1.0357592105865479,
 'monitor_decoder_conditioning_max': 1.2892680168151855}

## Train noise critic

In [5]:
latest_critic_pretrain_metrics = runtime.critic_pretrain()
latest_critic_pretrain_metrics

critic_pretrain:   0%|          | 0/2000 [00:00<?, ?it/s]

[critic_pretrain] step 1/2000: train: critic_loss=1.2028; monitor: critic_monitor_snapshot_score_norm_mean=145.2396, critic_monitor_transport_norm_mean=16.5768, critic_monitor_transport_norm_max=24.5900, critic_monitor_transport_t_min=0.1000, critic_monitor_transport_t_max=0.9900
[critic_pretrain] step 2000/2000: train: critic_loss=0.0765; monitor: critic_monitor_snapshot_score_norm_mean=43.6947, critic_monitor_transport_norm_mean=1.9034, critic_monitor_transport_norm_max=3.2245, critic_monitor_transport_t_min=0.1000, critic_monitor_transport_t_max=0.9900


{'train_critic_loss': 0.07647375017404556,
 'monitor_critic_monitor_snapshot_score_norm_mean': 43.694679260253906,
 'monitor_critic_monitor_transport_norm_mean': 1.9034069776535034,
 'monitor_critic_monitor_transport_norm_max': 3.2245476245880127,
 'monitor_critic_monitor_transport_t_min': 0.10000000149011612,
 'monitor_critic_monitor_transport_t_max': 0.9900000095367432}

## Integrated training

Integrated training is now delegated to the runtime. Each integrated step performs one chart-repair update, `n_critic_updates_every_transport_step` critic updates, then one chart-transport update. The integrated monitor stack logs constraint, critic, conditioning, and sampling monitors on the configured schedule.


In [ ]:
latest_integrated_metrics = runtime.integrated()
latest_integrated_metrics

integrated:   0%|          | 0/1000000 [00:00<?, ?it/s]

[integrated] step 1/1000000: train: critic=0.0752, repair=0.0003, data_cycle=0.0001, prior_cycle=0.0002, data_dual=0.0000, prior_dual=0.0000, transport=0.0003, enc_transport=0.0000, dec_transport=0.0003, field=1.8190, gen_ll=-60.6659; monitor: recon_err=0.0707, latent_norm=1.1814, score=43.3996, field=2.3595, enc_cond=1.6589, dec_cond=1.0458, gen_ll=-60.7839, gen_ll_std=124.6623
[integrated] step 2/1000000: train: critic=0.0805, repair=0.0008, data_cycle=0.0003, prior_cycle=0.0005, data_dual=0.0000, prior_dual=0.0000, transport=0.0009, enc_transport=0.0000, dec_transport=0.0009, field=1.5138, gen_ll=-58.8903; monitor: score=43.2377, field=1.9024
[integrated] step 5/1000000: train: critic=0.1272, repair=0.0444, data_cycle=0.0184, prior_cycle=0.0261, data_dual=0.0000, prior_dual=0.0000, transport=0.0114, enc_transport=0.0000, dec_transport=0.0113, field=2.4086, gen_ll=-56.0450; monitor: score=54.5200, field=4.3320
[integrated] step 10/1000000: train: critic=0.4453, repair=0.2246, data_cy